# Setup

Imports and such:

In [9]:
from importlib import reload

import numpy as np
import pyvista as pv

import skyvista as sv

import carlee_tools as ct

# Enable automatic reloading of edits to packages
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Ways to use skyvista from simplest to most complicated

## Simplest: `plot_gridded` and `plot_trajectories`

### tl;dr

In [5]:
reload(sv.plotter)
reload(sv.varspec)
reload(sv)

# Read in example data
ds = sv.load_example_storm_data()

# Calculate variable we'll use as the cold pool
ds["theta_deficit"] = (ds["THETA"].mean(["x", "y"]) - ds["THETA"]).where(
    ds["z"] <= 3000
)

# Make separate updraft and downdraft variables
ds["updraft"] = ds["WC"].where(ds["WC"] > 0)
ds["downdraft"] = -ds["WC"].where(ds["WC"] < 0)

# Make base-state relative winds
for wind_var in ["UC", "VC", "WC"]:
    ds[f"{wind_var}_bsr"] = ds[wind_var] - ds[wind_var].mean(["x", "y"])

# Now actually create the scene using the new API
scene = sv.plot_gridded(
    ds,
    contours={
        # Updraft contours
        "updraft": {
            "opacity": 0.4,
            "isosurfaces": np.arange(7, 25, 2),
            "cmap": ct.plotting.shifted_colormap("Greens", (0.3, 1.0)),
        },
        # Downdraft contours
        "downdraft": {
            "opacity": 0.4,
            "isosurfaces": np.arange(2, 8, 2),
            "cmap": ct.plotting.shifted_colormap("Oranges", (0.3, 1.0)),
        },
        # Cold pool
        "theta_deficit": {"opacity": 0.5, "isosurfaces": [1, 2], "cmap": "Blues"},
    },
    # 2D surface of accumulated rain
    slices={"ACCPR": {"cmap": "Blues"}},
    # Wind vectors
    vectors={
        "wind": {
            "u": "UC_bsr",
            "v": "VC_bsr",
            "w": "WC_bsr",
            "factor": 500,
            "tolerance": 0.05,
        }
    },
)

# Configure the plotter and show
scene.plotter.set_scale(zscale=3)
scene.plotter.camera_position = pv.CameraPosition(
    position=(201249.4941441632, 201249.4941441632, 49336.84095394194),
    focal_point=(110000.0, 110000.0, 18920.34290599823),
    viewup=(0.0, 0.0, 1.0),
)
scene.show()

Widget(value='<iframe src="http://localhost:34885/index.html?ui=P_0x7fcc9802b390_1&reconnect=auto" class="pyvi…

A view with name (P_0x7fcc9802b390_1) is already registered
 => returning previous one


Widget(value='<iframe src="http://localhost:34885/index.html?ui=P_0x7fcc9802b390_1&reconnect=auto" class="pyvi…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 192MB
Dimensions:        (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B ...
  * z              (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y              (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x              (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables:
    UC             (time, z, y, x) float32 17MB 0.6484 0.6469 ... 5.082 5.064
    VC             (time, z, y, x) float32 17MB 0.6276 0.6265 ... 0.04879
    WC             (time, z, y, x) float32 17MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    THETA          (time, z, y, x) float32 17MB 301.9 301.9 ... 517.4 517.4
    R_condensate   (time, z, y, x) float32 17MB ...
    ACCPR          (time, y, x) float32 162kB ...
    theta_deficit  (time, z, y, x) float32 17MB -0.9074 -0.9091 ... nan

### Walkthrough

`plot_gridded` is the simplest way to take a quick look at something, or setup a simple scene
from a single dataset. It returns a `Scene` object that you can further customize or render. First let's load the example data:

In [10]:
ds = sv.load_example_storm_data()
ds

<xarray.Dataset> Size: 87MB
Dimensions:       (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time          (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes     (time) float64 8B ...
  * z             (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y             (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x             (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables:
    UC            (time, z, y, x) float32 17MB ...
    VC            (time, z, y, x) float32 17MB ...
    WC            (time, z, y, x) float32 17MB ...
    THETA         (time, z, y, x) float32 17MB ...
    R_condensate  (time, z, y, x) float32 17MB ...
    ACCPR         (time, y, x) float32 162kB ...

UC, VC, and WC are the x/y/z components of the wind, THETA is the potential temperature, R_condensate is the mass mixing ratio of all hydrometeors, and ACCPR is the cumulative amount of rain in a surface gridpoint. Now 

In [11]:
# Create a scene with contours and show it
_ = sv.plot_gridded(
    # Pass your (potentially non-uniformly) gridded data
    ds,
    contours={"WC": {}},
)

Widget(value='<iframe src="http://localhost:34885/index.html?ui=P_0x7fcc84720a50_3&reconnect=auto" class="pyvi…

Note the form of what we pass to `plot_gridded`: 
- Keyword argument of the type of object we want to create in the scene (contours of isosurfaces)
- Value of this argument is a dict where:
  - Each key is the name of a variable in the dataset and,
  - Each value is a dict of plotting settings for that variable
  
In the example above, we passed an empty dict for the `WC` plotting settings to just use all default options.

This worked, but the visualization isn't that useful, since we can only see the outermost isosurface. Nonetheless, we can see that the scene is interactive and shows the general shape of the storm and its cold pool. We can give all of the isosurfaces a uniform semi-transparent opacity to make this more useful:

In [12]:
scene = sv.plot_gridded(
    ds,
    contours={"WC": {"opacity": 0.4}},
)
scene.show()

Widget(value='<iframe src="http://localhost:34885/index.html?ui=P_0x7fcc847211d0_4&reconnect=auto" class="pyvi…

A view with name (P_0x7fcc847211d0_4) is already registered
 => returning previous one


Widget(value='<iframe src="http://localhost:34885/index.html?ui=P_0x7fcc847211d0_4&reconnect=auto" class="pyvi…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 87MB
Dimensions:       (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time          (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes     (time) float64 8B ...
  * z             (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y             (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x             (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables:
    UC            (time, z, y, x) float32 17MB ...
    VC            (time, z, y, x) float32 17MB ...
    WC            (time, z, y, x) float32 17MB ...
    THETA         (time, z, y, x) float32 17MB ...
    R_condensate  (time, z, y, x) float32 17MB ...
    ACCPR         (time, y, x) float32 162kB ..., ContourSpec(name='contour_WC', empty_ok=False, pyvista_create_kwargs={}, pyvista_add_kwargs={}, _geometry=ContourGeometry(varname='WC', isosurfaces=None, scalar=None, individual_meshe

Now we can see the core of the upward vertical motion much more clearly. However, we often want to see only the updraft, not the downdraft. A few ways of achieving this:

In [6]:
scene = sv.plot_gridded(
    # Simplest option: manipulate and/or subset your data
    ds.where(ds["WC"] > 0),
    contours={"WC": {"opacity": 0.4}},
)
scene.show()

Widget(value='<iframe src="http://localhost:46209/index.html?ui=P_0x7fda80f39a90_3&reconnect=auto" class="pyvi…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 209MB
Dimensions:        (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B 0.0
  * z              (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y              (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x              (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables:
    UC             (time, z, y, x) float32 17MB nan nan nan nan ... nan nan nan
    VC             (time, z, y, x) float32 17MB nan nan nan nan ... nan nan nan
    WC             (time, z, y, x) float32 17MB nan nan nan nan ... nan nan nan
    THETA          (time, z, y, x) float32 17MB nan nan nan nan ... nan nan nan
    R_condensate   (time, z, y, x) float32 17MB nan nan nan nan ... nan nan nan
    ACCPR          (time, y, x, z) float32 17MB nan 0.0 0.0 0.0 ... nan nan nan
 

In [7]:
# For contours, another option is specifying isosurface values directly via the
# "isosurfaces" keyword argument
# `where` calls can be slow on large xarray Datasets, so this is probably preferable
# if the subsetting you want can be easily expressed this way

scene = sv.plot_gridded(
    ds,
    contours={"WC": {"opacity": 0.4, "isosurfaces": np.arange(1, 25, 2)}},
)
scene.show()

Widget(value='<iframe src="http://localhost:46209/index.html?ui=P_0x7fda80f38cd0_4&reconnect=auto" class="pyvi…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 192MB
Dimensions:        (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B ...
  * z              (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y              (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x              (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables:
    UC             (time, z, y, x) float32 17MB 0.6484 0.6469 ... 5.082 5.064
    VC             (time, z, y, x) float32 17MB 0.6276 0.6265 ... 0.04879
    WC             (time, z, y, x) float32 17MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    THETA          (time, z, y, x) float32 17MB 301.9 301.9 ... 517.4 517.4
    R_condensate   (time, z, y, x) float32 17MB ...
    ACCPR          (time, y, x) float32 162kB ...
    theta_deficit  (time, z, y, x) float32 17MB -0.9074 -0.9091 ... nan

We'll go with the second approach for now. For a continuous variable like velocity, we may want to color this such that only the intensity of the color corresponds with the value, rather than changing the hue as in a colormap like `viridis`:

In [8]:
# Use a sequential colormap where intensity represents the value

scene = sv.plot_gridded(
    ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(1, 25, 2), "cmap": "Greens"}
    },
)
scene.show()

Widget(value='<iframe src="http://localhost:46209/index.html?ui=P_0x7fda80f3a990_5&reconnect=auto" class="pyvi…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 192MB
Dimensions:        (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B ...
  * z              (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y              (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x              (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables:
    UC             (time, z, y, x) float32 17MB 0.6484 0.6469 ... 5.082 5.064
    VC             (time, z, y, x) float32 17MB 0.6276 0.6265 ... 0.04879
    WC             (time, z, y, x) float32 17MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    THETA          (time, z, y, x) float32 17MB 301.9 301.9 ... 517.4 517.4
    R_condensate   (time, z, y, x) float32 17MB ...
    ACCPR          (time, y, x) float32 162kB ...
    theta_deficit  (time, z, y, x) float32 17MB -0.9074 -0.9091 ... nan

Looking much better. It's simple to show multiple variables; let's add a 2D surface of the accumulated rainfall with the `slices` argument:

In [9]:
scene = sv.plot_gridded(
    ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(1, 25, 2), "cmap": "Greens"}
    },
    slices={"ACCPR": {"cmap": "Blues"}},
)
scene.show()

Widget(value='<iframe src="http://localhost:46209/index.html?ui=P_0x7fda80f3b250_6&reconnect=auto" class="pyvi…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 192MB
Dimensions:        (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B ...
  * z              (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y              (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x              (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables:
    UC             (time, z, y, x) float32 17MB 0.6484 0.6469 ... 5.082 5.064
    VC             (time, z, y, x) float32 17MB 0.6276 0.6265 ... 0.04879
    WC             (time, z, y, x) float32 17MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    THETA          (time, z, y, x) float32 17MB 301.9 301.9 ... 517.4 517.4
    R_condensate   (time, z, y, x) float32 17MB ...
    ACCPR          (time, y, x) float32 162kB ...
    theta_deficit  (time, z, y, x) float32 17MB -0.9074 -0.9091 ... nan

And the cold pool, which we'll calculate as the low-level potential temperature deficit relative to the layer-average value:

In [10]:
ds["theta_lr"] = (ds["THETA"].mean(["x", "y"]) - ds["THETA"]).where(ds["z"] <= 3000)

scene = sv.plot_gridded(
    ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(1, 25, 2), "cmap": "Greens"},
        "theta_lr": {"opacity": 0.5, "isosurfaces": [1, 2], "cmap": "Blues"},
    },
    slices={"ACCPR": {"cmap": "Blues"}},
)
scene.show()

Widget(value='<iframe src="http://localhost:46209/index.html?ui=P_0x7fda80f3bed0_7&reconnect=auto" class="pyvi…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 210MB
Dimensions:        (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B ...
  * z              (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y              (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x              (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables: (12/13)
    UC             (time, z, y, x) float32 17MB 0.6484 0.6469 ... 5.082 5.064
    VC             (time, z, y, x) float32 17MB 0.6276 0.6265 ... 0.04879
    WC             (time, z, y, x) float32 17MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    THETA          (time, z, y, x) float32 17MB 301.9 301.9 ... 517.4 517.4
    R_condensate   (time, z, y, x) float32 17MB ...
    ACCPR          (time, y, x) float32 162kB ...
    ...             ...
    updraft        (time, z, y, x) floa

Nice, but sort of hard to see. Let's do two things: first, let's limit the updraft contours to larger values of W to see it more clearly and remove some of the noise from the vertical motion at the leading edge of the cold pool. Second, let's zoom in. 

The `Scene` object gives you access to the underlying PyVista `Plotter` via `.plotter`. The camera position can be gotten or set via the `.camera_position` attribute. Here's how to set the camera position:

In [14]:
ds["theta_lr"] = (ds["THETA"].mean(["x", "y"]) - ds["THETA"]).where(ds["z"] <= 3000)

# Create the scene
scene = sv.plot_gridded(
    ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(5, 25, 2), "cmap": "Greens"},
        "theta_lr": {"opacity": 0.5, "isosurfaces": [1, 2], "cmap": "Blues"},
    },
    slices={"ACCPR": {"cmap": "Blues"}},
)

# Access the plotter via scene.plotter
scene.show()

Widget(value='<iframe src="http://localhost:34885/index.html?ui=P_0x7fcc2c60da90_6&reconnect=auto" class="pyvi…

A view with name (P_0x7fcc2c60da90_6) is already registered
 => returning previous one


Widget(value='<iframe src="http://localhost:34885/index.html?ui=P_0x7fcc2c60da90_6&reconnect=auto" class="pyvi…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 105MB
Dimensions:       (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time          (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes     (time) float64 8B ...
  * z             (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y             (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x             (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables:
    UC            (time, z, y, x) float32 17MB ...
    VC            (time, z, y, x) float32 17MB ...
    WC            (time, z, y, x) float32 17MB ...
    THETA         (time, z, y, x) float32 17MB 301.9 301.9 301.9 ... 517.4 517.4
    R_condensate  (time, z, y, x) float32 17MB ...
    ACCPR         (time, y, x) float32 162kB ...
    theta_lr      (time, z, y, x) float32 17MB -0.9074 -0.9091 ... nan nan, ContourSpec(name='contour_WC_iso5', empty_ok=False, pyvista_create_kwargs={},

Now we can get the camera position from the scene's plotter object. Try running all of the above cells up to this point, then alternate between moving around in the interactive scene directly above this cell and running the following cell, and see how the value it returns changes:

In [12]:
scene.plotter.camera_position

CameraPosition(position=(318949.7503599592, 318949.7503599592, 103882.97893143933),
               focal_point=(110000.0, 110000.0, 34233.06214427948),
               viewup=(0.0, 0.0, 1.0))

Now if we set the `.camera_position` attribute of the plotter to this value, anything we plot will start at this camera angle:

In [13]:
# Create the scene
scene = sv.plot_gridded(
    ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(5, 25, 2), "cmap": "Greens"},
        "theta_lr": {"opacity": 0.5, "isosurfaces": [1, 2], "cmap": "Blues"},
    },
    slices={"ACCPR": {"cmap": "Blues"}},
)

# Set the camera position, based on the value we got by interactively moving around
# the figure above and copy-pasting the output of the code cell above this one
# You can set the camera position before or after calling show()
scene.plotter.camera_position = pv.CameraPosition(
    position=(201249.4941441632, 201249.4941441632, 49336.84095394194),
    focal_point=(110000.0, 110000.0, 18920.34290599823),
    viewup=(0.0, 0.0, 1.0),
)

scene.show()

Widget(value='<iframe src="http://localhost:46209/index.html?ui=P_0x7fda80f3a850_9&reconnect=auto" class="pyvi…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 210MB
Dimensions:        (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B ...
  * z              (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y              (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x              (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables: (12/13)
    UC             (time, z, y, x) float32 17MB 0.6484 0.6469 ... 5.082 5.064
    VC             (time, z, y, x) float32 17MB 0.6276 0.6265 ... 0.04879
    WC             (time, z, y, x) float32 17MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    THETA          (time, z, y, x) float32 17MB 301.9 301.9 ... 517.4 517.4
    R_condensate   (time, z, y, x) float32 17MB ...
    ACCPR          (time, y, x) float32 162kB ...
    ...             ...
    updraft        (time, z, y, x) floa

The `Scene` object opens up further options. Let's say we want to show both the updraft and downdraft, but with different colormaps and max/min values. Diverging colormaps with different max/min magnitudes are tricky in pyvista; there is no analog to the `TwoSlopeNorm` that can be used to accomplish this in matplotlib. You could create separate variables for each, but we already saw that `.where` operations can be expensive on large xarray datasets. We could create two sets of contours with different isosurface values and colormaps, but we can't pass a dictionary with two "WC" keys. With the `Scene` API, we can add contours incrementally:

In [ ]:
# Create a new scene
scene = sv.Scene()

# Add the updraft contours
scene.add_contour(ds, "WC", opacity=0.4, isosurfaces=np.arange(5, 25, 2), cmap="Greens")

# Add cold pool and surface rain
scene.add_contour(ds, "theta_lr", opacity=0.5, isosurfaces=[1, 2], cmap="Blues")
scene.add_slice(ds, "ACCPR", cmap="Blues")

# Now add the downdraft with different settings - same variable, different coloring
scene.add_contour(
    ds, "WC", opacity=0.4, isosurfaces=np.arange(-8, -2, 2), cmap="Reds_r"
)

# Set camera and show
scene.plotter.camera_position = pv.CameraPosition(
    position=(201249.4941441632, 201249.4941441632, 49336.84095394194),
    focal_point=(110000.0, 110000.0, 18920.34290599823),
    viewup=(0.0, 0.0, 1.0),
)
scene.show()

Widget(value='<iframe src="http://localhost:46209/index.html?ui=P_0x7fda7f2f4a50_10&reconnect=auto" class="pyv…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 210MB
Dimensions:        (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B ...
  * z              (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y              (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x              (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables: (12/13)
    UC             (time, z, y, x) float32 17MB 0.6484 0.6469 ... 5.082 5.064
    VC             (time, z, y, x) float32 17MB 0.6276 0.6265 ... 0.04879
    WC             (time, z, y, x) float32 17MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    THETA          (time, z, y, x) float32 17MB 301.9 301.9 ... 517.4 517.4
    R_condensate   (time, z, y, x) float32 17MB ...
    ACCPR          (time, y, x) float32 162kB ...
    ...             ...
    updraft        (time, z, y, x) floa

A common problem when plotting contours in matplotlib that is especially prevalent in pyvista is that the sequential colormaps included with matplotlib all start at very light colors, so the lowest contour is sometimes very nearly white, as we can see in the above. A tool for dealing with this is the `shifted_colormap` function from the `carlee_tools` package:

In [ ]:
# Create scene with shifted colormap to avoid light colors at low values
scene = sv.Scene()

# Note the colormap we use for WC here - using carlee_tools to shift it
scene.add_contour(
    ds,
    "WC",
    opacity=0.4,
    isosurfaces=np.arange(5, 25, 2),
    # Use carlee_tools to create a colormap that starts 30% of the way
    # through the normal matplotlib `Greens` colormap
    cmap=ct.plotting.shifted_colormap("Greens", (0.3, 1)),
)
scene.add_contour(ds, "theta_lr", opacity=0.5, isosurfaces=[1, 2], cmap="Blues")
scene.add_slice(ds, "ACCPR", cmap="Blues")

# Uncomment to add downdraft with shifted red colormap:
# scene.add_contour(
#     ds, "WC",
#     opacity=0.4,
#     isosurfaces=np.arange(-8, -2, 2),
#     # Analogously to the updraft, but for a reversed colormap
#     cmap=ct.plotting.shifted_colormap("Reds_r", (0, 0.5)),
# )

scene.plotter.camera_position = pv.CameraPosition(
    position=(201249.4941441632, 201249.4941441632, 49336.84095394194),
    focal_point=(110000.0, 110000.0, 18920.34290599823),
    viewup=(0.0, 0.0, 1.0),
)
scene.show()

Widget(value='<iframe src="http://localhost:46209/index.html?ui=P_0x7fda7f2f60d0_11&reconnect=auto" class="pyv…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 210MB
Dimensions:        (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B ...
  * z              (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y              (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x              (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables: (12/13)
    UC             (time, z, y, x) float32 17MB 0.6484 0.6469 ... 5.082 5.064
    VC             (time, z, y, x) float32 17MB 0.6276 0.6265 ... 0.04879
    WC             (time, z, y, x) float32 17MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    THETA          (time, z, y, x) float32 17MB 301.9 301.9 ... 517.4 517.4
    R_condensate   (time, z, y, x) float32 17MB ...
    ACCPR          (time, y, x) float32 162kB ...
    ...             ...
    updraft        (time, z, y, x) floa

In [16]:
# Example: subsetting data by stride to reduce complexity
xy_stride = 25
z_stride = 10
ds.isel({
    "x": slice(None, None, xy_stride),
    "y": slice(None, None, xy_stride),
    "z": slice(None, None, z_stride),
})

<xarray.Dataset> Size: 43kB
Dimensions:        (time: 1, z: 11, y: 9, x: 9)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B ...
  * z              (z) float64 88B -11.63 368.3 ... 1.858e+04 2.108e+04
  * y              (y) float64 72B 6e+04 7.25e+04 8.5e+04 ... 1.475e+05 1.6e+05
  * x              (x) float64 72B 6e+04 7.25e+04 8.5e+04 ... 1.475e+05 1.6e+05
Data variables: (12/13)
    UC             (time, z, y, x) float32 4kB 0.6484 0.3754 ... 5.391 4.713
    VC             (time, z, y, x) float32 4kB 0.6276 0.5607 ... 0.573 -0.3852
    WC             (time, z, y, x) float32 4kB 0.0 0.0 0.0 ... 0.02639 -0.1761
    THETA          (time, z, y, x) float32 4kB 301.9 301.9 301.9 ... 486.4 486.1
    R_condensate   (time, z, y, x) float32 4kB ...
    ACCPR          (time, y, x) float32 324B ...
    ...             ...
    updraft        (time, z, y, x) float32 4kB nan nan nan ... 0.02639 nan
    downdraft      (time, z, y, x) float32 4kB nan nan nan ... nan nan 0.1761
    UC_bsr         (time, z, y, x) float32 4kB 0.7014 0.4284 ... 0.3357 -0.3421
    VC_bsr         (time, z, y, x) float32 4kB 0.6251 0.5582 ... 0.5749 -0.3833
    WC_bsr         (time, z, y, x) float32 4kB 0.0 0.0 0.0 ... 0.03173 -0.1708
    theta_lr       (time, z, y, x) float32 4kB -0.9074 -0.9093 ... nan nan

In [17]:
reload(sv.plotter)
reload(sv.convenience)
reload(sv)

scene = sv.plot_gridded(
    ds.sel({"y": slice(0, 110_000)}).where(ds["WC"] >= 1),
    volumes={"WC": {}},
)
scene.show()

Widget(value='<iframe src="http://localhost:46209/index.html?ui=P_0x7fda95868cd0_12&reconnect=auto" class="pyv…

Scene(background='#f8f6f1', title=None, show_grid=True, _specs=[(<xarray.Dataset> Size: 114MB
Dimensions:        (time: 1, z: 108, y: 101, x: 201)
Coordinates:
  * time           (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes      (time) float64 8B 0.0
  * z              (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y              (y) float64 808B 6e+04 6.05e+04 6.1e+04 ... 1.095e+05 1.1e+05
  * x              (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables: (12/13)
    UC             (time, z, y, x) float32 9MB nan nan nan nan ... nan nan nan
    VC             (time, z, y, x) float32 9MB nan nan nan nan ... nan nan nan
    WC             (time, z, y, x) float32 9MB nan nan nan nan ... nan nan nan
    THETA          (time, z, y, x) float32 9MB nan nan nan nan ... nan nan nan
    R_condensate   (time, z, y, x) float32 9MB nan nan nan nan ... nan nan nan
    ACCPR          (time, y, x, z) float32 9MB nan nan nan nan ... nan nan na

In [ ]:
reload(sv.plotter)
reload(sv.convenience)
reload(sv.core)
reload(sv)

scene = sv.plot_gridded(
    ds,
    contours={"WC": {"opacity": 0.3, "cmap": "Greens", "isosurfaces": [1, 3, 5, 10]}},
    volumes={
        "R_condensate": {
            "cmap": "Blues",
            "opacity": pv.opacity_transfer_function([20, 35, 60, 90, 250], 256),
            "clim": [1e-5, ds["R_condensate"].max().values],
        }
    },
    vectors={
        "wind": {"u": "UC", "v": "VC", "w": "WC", "factor": 500, "tolerance": 0.05}
    },
)
scene.show()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
# Export the scene to HTML for sharing
# scene.export_html("/home/cmdavis4/projects/sean_students/test_ic_export.html")